# KV Cache：从增量注意力到分页与连续批处理

本 notebook 用一个最小 self-attention 实现验证 full forward 与逐 token cache forward 等价，再用小型分配器模拟 PagedAttention 的 block table、共享前缀与 copy-on-write。

## 学习目标

1. 说清 KV Cache 缓存什么、为什么不缓存历史 Q，以及复杂度如何变化。
2. 掌握逻辑形状 `[layers, 2, tokens, kv_heads, head_dim]` 与显存公式。
3. 从零实现增量 append，并验证 full forward 与 cached forward 对齐。
4. 理解 MHA/MQA/GQA、分页、prefix sharing、COW、量化与 offload 的取舍。
5. 能分析 prefill/decode、continuous batching、chunked prefill、TTFT/TPOT/ITL。


In [ ]:
import math  # 导入本单元所需的依赖。
from collections import defaultdict, deque  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。

torch.manual_seed(23)  # 执行当前语句以推进本节示例。
torch.set_printoptions(precision=5, sci_mode=False)  # 计算并保存当前步骤的中间状态。
print("PyTorch:", torch.__version__)  # 执行当前语句以推进本节示例。


## 1. 缓存对象、增量公式与复杂度

第 l 层对历史 token 产生 `K=XWk`、`V=XWv`。第 t 个新 token 只计算当前 `q_t,k_t,v_t`：

\[o_t=\operatorname{softmax}(q_t[K_{1:t-1};k_t]^T/\sqrt{d_h})[V_{1:t-1};v_t].\]

历史 K/V 会被未来每个 query 重复读取，历史 Q 只在自己的步骤使用一次，因此通常不缓存 Q。无 cache 时反复 full forward，attention 总量可近似到 `O(T^3 d)`；cache 后第 t 步是单 query 扫 t 个 keys，总 attention 为 `O(T^2 d)`，且历史投影与 FFN 不再重算。单步仍随历史长度线性读取，cache 是空间换重复计算，不是 O(1) 魔法。


## 2. 形状与显存公式

逻辑上 K 或 V 常写成 `[B,Hkv,S,Dh]`，全模型可概念化为 `[L,2,B,Hkv,S,Dh]`。分页服务把 B、S 映射成总物理 token blocks。主数据字节数：

\[M=2\cdot L\cdot N_{tokens}\cdot H_{kv}\cdot D_h\cdot bytes.\]

GQA/MQA 的 `Hkv` 小于 query head 数；计算容量时最常见的错误就是误用 `Hq`，以及漏掉 K、V 两份的因子 2。


In [ ]:
def kv_cache_bytes(layers, tokens, kv_heads, head_dim, bytes_per_element=2):  # 定义本节可复用的核心函数。
    return 2 * layers * tokens * kv_heads * head_dim * bytes_per_element  # 返回当前分支计算出的结果。

def gib(n_bytes):  # 定义本节可复用的核心函数。
    return n_bytes / 1024**3  # 返回当前分支计算出的结果。

layers, kv_heads, head_dim = 40, 8, 128  # 计算并保存当前步骤的中间状态。
per_token = kv_cache_bytes(layers, 1, kv_heads, head_dim, 2)  # 计算并保存当前步骤的中间状态。
total = kv_cache_bytes(layers, 12 * 8192, kv_heads, head_dim, 2)  # 计算并保存当前步骤的中间状态。
print(f"每 token: {per_token / 1024:.1f} KiB")  # 执行当前语句以推进本节示例。
print(f"12 个 8192-token 请求: {gib(total):.2f} GiB")  # 执行当前语句以推进本节示例。
print(f"若误用 32 个 Q heads: {gib(kv_cache_bytes(layers, 12*8192, 32, head_dim, 2)):.2f} GiB")  # 执行当前语句以推进本节示例。
assert gib(total) == 15.0  # 用受控断言验证关键不变量。


## 3. 从零实现：full attention 与增量 cache 对齐

下面的模块只演示单层 MHA，不含 RoPE。`forward_full` 一次计算整段 causal attention；`forward_step` 接收一个新 token，把 K/V 追加到 tuple cache。逐步输出应与 full 输出逐位置一致，这是所有 cache 实现最重要的 golden test。


In [ ]:
class CacheableSelfAttention(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, d_model=16, n_heads=4):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        assert d_model % n_heads == 0  # 用受控断言验证关键不变量。
        self.n_heads = n_heads  # 计算并保存当前步骤的中间状态。
        self.head_dim = d_model // n_heads  # 计算并保存当前步骤的中间状态。
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)  # 计算并保存当前步骤的中间状态。
        self.out_proj = nn.Linear(d_model, d_model, bias=False)  # 计算并保存当前步骤的中间状态。

    def _split_qkv(self, x):  # 定义本节可复用的核心函数。
        b, s, d = x.shape  # 计算并保存当前步骤的中间状态。
        qkv = self.qkv(x).view(b, s, 3, self.n_heads, self.head_dim)  # 计算并保存当前步骤的中间状态。
        q, k, v = qkv.unbind(dim=2)  # 计算并保存当前步骤的中间状态。
        return tuple(t.transpose(1, 2) for t in (q, k, v))  # 返回当前分支计算出的结果。

    def _merge(self, x):  # 定义本节可复用的核心函数。
        b, h, s, dh = x.shape  # 计算并保存当前步骤的中间状态。
        return self.out_proj(x.transpose(1, 2).contiguous().view(b, s, h * dh))  # 返回当前分支计算出的结果。

    def forward_full(self, x):  # 定义本节可复用的核心函数。
        q, k, v = self._split_qkv(x)  # 计算并保存当前步骤的中间状态。
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)  # 计算并保存当前步骤的中间状态。
        causal = torch.ones(x.size(1), x.size(1), dtype=torch.bool, device=x.device).triu(1)  # 计算并保存当前步骤的中间状态。
        scores = scores.masked_fill(causal, float("-inf"))  # 计算并保存当前步骤的中间状态。
        probs = torch.softmax(scores.float(), dim=-1).to(x.dtype)  # 计算并保存当前步骤的中间状态。
        return self._merge(probs @ v)  # 返回当前分支计算出的结果。

    def forward_step(self, x_new, cache=None):  # 定义本节可复用的核心函数。
        assert x_new.size(1) == 1  # 用受控断言验证关键不变量。
        q, k_new, v_new = self._split_qkv(x_new)  # 计算并保存当前步骤的中间状态。
        if cache is None:  # 按当前条件选择后续控制路径。
            k_all, v_all = k_new, v_new  # 计算并保存当前步骤的中间状态。
        else:  # 处理前置条件不成立的分支。
            k_all = torch.cat((cache[0], k_new), dim=-2)  # 计算并保存当前步骤的中间状态。
            v_all = torch.cat((cache[1], v_new), dim=-2)  # 计算并保存当前步骤的中间状态。
        scores = q @ k_all.transpose(-2, -1) / math.sqrt(self.head_dim)  # 计算并保存当前步骤的中间状态。
        probs = torch.softmax(scores.float(), dim=-1).to(x_new.dtype)  # 计算并保存当前步骤的中间状态。
        return self._merge(probs @ v_all), (k_all, v_all)  # 返回当前分支计算出的结果。

attn = CacheableSelfAttention().eval()  # 计算并保存当前步骤的中间状态。
x = torch.randn(2, 7, 16)  # 计算并保存当前步骤的中间状态。
full_out = attn.forward_full(x)  # 计算并保存当前步骤的中间状态。
cache, step_outputs = None, []  # 计算并保存当前步骤的中间状态。
for t in range(x.size(1)):  # 遍历输入元素以累积或检查结果。
    out_t, cache = attn.forward_step(x[:, t:t+1], cache)  # 计算并保存当前步骤的中间状态。
    step_outputs.append(out_t)  # 执行当前语句以推进本节示例。
cached_out = torch.cat(step_outputs, dim=1)  # 计算并保存当前步骤的中间状态。
print("full/cached shape:", full_out.shape, cached_out.shape)  # 执行当前语句以推进本节示例。
print("K/V cache shape:", cache[0].shape, cache[1].shape)  # 执行当前语句以推进本节示例。
print("最大输出误差:", (full_out - cached_out).abs().max().item())  # 执行当前语句以推进本节示例。
assert torch.allclose(full_out, cached_out, atol=2e-6)  # 用受控断言验证关键不变量。


### 张量形状逐步追踪

本例 `B=2,H=4,Dh=4`。prefill 长度 7 时 K/V 为 `[2,4,7,4]`；第 8 个 token 的 Q 为 `[2,4,1,4]`，与追加后的 K `[2,4,8,4]` 相乘，score 为 `[2,4,1,8]`。在单 token decode 中，只有一个 query，传入的有效历史长度天然防止看未来；一次验证多个 speculative token 时仍需块内 causal mask。

上面的 tuple + `torch.cat` 每步都会复制历史，只用于教学。生产实现预分配或分页写 slot，避免 O(S²) 的 host-side 拼接搬运。


In [ ]:
x_next = torch.randn(2, 1, 16)  # 计算并保存当前步骤的中间状态。
q_next, k_next, v_next = attn._split_qkv(x_next)  # 计算并保存当前步骤的中间状态。
k_after_append = torch.cat((cache[0], k_next), dim=-2)  # 计算并保存当前步骤的中间状态。
score_shape = (q_next @ k_after_append.transpose(-2, -1)).shape  # 计算并保存当前步骤的中间状态。
for name, shape in {"q_new": q_next.shape, "k_new": k_next.shape,  # 遍历输入元素以累积或检查结果。
                    "k_all": k_after_append.shape, "scores": score_shape}.items():  # 执行当前语句以推进本节示例。
    print(f"{name:8s} -> {tuple(shape)}")  # 执行当前语句以推进本节示例。


## 4. MHA、GQA、MQA：容量和带宽的直接关系

MHA 的 `Hkv=Hq`；GQA 让若干 Q heads 共享一个 KV head；MQA 的 `Hkv=1`。Q heads 可以仍有各自查询投影，因此 cache 节省来自 K/V 头变少，而不是 model hidden size 同比例变小。内核应直接按 group 映射，若物理 repeat K/V 回 Hq 会丢掉带宽优势。


In [ ]:
Hq, sequence = 32, 32768  # 计算并保存当前步骤的中间状态。
for name, hkv in [("MHA", 32), ("GQA", 8), ("MQA", 1)]:  # 遍历输入元素以累积或检查结果。
    size = kv_cache_bytes(layers=32, tokens=sequence, kv_heads=hkv, head_dim=128, bytes_per_element=2)  # 计算并保存当前步骤的中间状态。
    group = Hq // hkv  # 计算并保存当前步骤的中间状态。
    print(f"{name}: Hkv={hkv:2d}, 每个 KV head 服务 {group:2d} 个 Q heads, cache={gib(size):.2f} GiB")  # 计算并保存当前步骤的中间状态。


## 5. 最小分页分配器：block table、共享与 COW

连续按最大长度预留会浪费；按需扩展又可能搬迁和外部碎片。分页把物理池切成固定 token blocks，请求的逻辑块可映射到任意物理块。完整前缀块可由多个请求共享；写入仍被共享的尾块前必须 copy-on-write。下例只模拟元数据，不存真实 KV。


In [ ]:
class MiniPagedAllocator:  # 定义承载本节状态与行为的数据结构。
    def __init__(self, num_blocks, block_size):  # 定义本节可复用的核心函数。
        self.block_size = block_size  # 计算并保存当前步骤的中间状态。
        self.free = deque(range(num_blocks))  # 计算并保存当前步骤的中间状态。
        self.refcount = defaultdict(int)  # 计算并保存当前步骤的中间状态。
        self.tables = {}  # 计算并保存当前步骤的中间状态。
        self.lengths = {}  # 计算并保存当前步骤的中间状态。

    def create(self, request_id):  # 定义本节可复用的核心函数。
        self.tables[request_id] = []  # 计算并保存当前步骤的中间状态。
        self.lengths[request_id] = 0  # 计算并保存当前步骤的中间状态。

    def _new_block(self):  # 定义本节可复用的核心函数。
        if not self.free:  # 按当前条件选择后续控制路径。
            raise RuntimeError("KV block pool exhausted")  # 遇到非法合同立即显式失败。
        block = self.free.popleft()  # 计算并保存当前步骤的中间状态。
        self.refcount[block] = 1  # 计算并保存当前步骤的中间状态。
        return block  # 返回当前分支计算出的结果。

    def append_one(self, request_id):  # 定义本节可复用的核心函数。
        length = self.lengths[request_id]  # 计算并保存当前步骤的中间状态。
        if length % self.block_size == 0:  # 按当前条件选择后续控制路径。
            self.tables[request_id].append(self._new_block())  # 执行当前语句以推进本节示例。
        elif self.refcount[self.tables[request_id][-1]] > 1:  # 按当前条件选择后续控制路径。
            old = self.tables[request_id][-1]  # 计算并保存当前步骤的中间状态。
            new = self._new_block()  # 真实系统还要复制尾块已有 KV
            self.refcount[old] -= 1  # 计算并保存当前步骤的中间状态。
            self.tables[request_id][-1] = new  # 计算并保存当前步骤的中间状态。
        self.lengths[request_id] += 1  # 计算并保存当前步骤的中间状态。

    def append(self, request_id, count):  # 定义本节可复用的核心函数。
        for _ in range(count):  # 遍历输入元素以累积或检查结果。
            self.append_one(request_id)  # 执行当前语句以推进本节示例。

    def share(self, source_id, new_id):  # 定义本节可复用的核心函数。
        self.tables[new_id] = list(self.tables[source_id])  # 计算并保存当前步骤的中间状态。
        self.lengths[new_id] = self.lengths[source_id]  # 计算并保存当前步骤的中间状态。
        for block in self.tables[new_id]:  # 遍历输入元素以累积或检查结果。
            self.refcount[block] += 1  # 计算并保存当前步骤的中间状态。

    def release(self, request_id):  # 定义本节可复用的核心函数。
        for block in self.tables.pop(request_id):  # 遍历输入元素以累积或检查结果。
            self.refcount[block] -= 1  # 计算并保存当前步骤的中间状态。
            if self.refcount[block] == 0:  # 按当前条件选择后续控制路径。
                self.free.append(block)  # 执行当前语句以推进本节示例。
        self.lengths.pop(request_id)  # 执行当前语句以推进本节示例。

pool = MiniPagedAllocator(num_blocks=8, block_size=4)  # 计算并保存当前步骤的中间状态。
pool.create("A")  # 执行当前语句以推进本节示例。
pool.append("A", 6)  # 执行当前语句以推进本节示例。
pool.share("A", "B")  # 执行当前语句以推进本节示例。
print("共享后 A/B tables:", pool.tables, "refcount:", dict(pool.refcount))  # 执行当前语句以推进本节示例。
pool.append_one("A")  # 尾块部分已用且共享，触发 COW
pool.append_one("B")  # 执行当前语句以推进本节示例。
print("分叉写入后 tables:", pool.tables, "refcount:", dict(pool.refcount))  # 执行当前语句以推进本节示例。
assert pool.tables["A"][0] == pool.tables["B"][0]  # 用受控断言验证关键不变量。
assert pool.tables["A"][-1] != pool.tables["B"][-1]  # 用受控断言验证关键不变量。


### 分页系统的状态不变量

逻辑位置通过 `logical_block=t//block_size` 和 `offset=t%block_size` 查表；物理 block id 不是 position id。allocator 只有 refcount 归零才能回收块。请求取消、beam prune、speculative rollback 都必须幂等更新表和引用。PagedAttention 优化的是分配、碎片与共享，不改变 attention 数学复杂度；FlashAttention 则优化 attention 中间矩阵的 IO，两者解决不同问题。


## 6. KV Cache 量化：动态激活而不是权重

对称 INT8 可按 token/head/group 计算 `scale=max(abs(x))/127`，保存 int8 主数据和 scale。粒度越细误差越小、元数据越多。K 的误差会经 QK 和 softmax 改变路由，V 的误差在线性加权中传播，因此两者可能需要不同量化轴。生产 kernel 应融合反量化，避免先展开成 FP16 临时 cache。


In [ ]:
def quantize_int8_per_vector(x, eps=1e-8):  # 定义本节可复用的核心函数。
    scale = x.abs().amax(dim=-1, keepdim=True).clamp_min(eps) / 127.0  # 计算并保存当前步骤的中间状态。
    q = torch.round(x / scale).clamp(-127, 127).to(torch.int8)  # 计算并保存当前步骤的中间状态。
    return q, scale  # 返回当前分支计算出的结果。

def dequantize_int8(q, scale):  # 定义本节可复用的核心函数。
    return q.float() * scale  # 返回当前分支计算出的结果。

kv = torch.randn(2, 8, 128, 64)  # 计算并保存当前步骤的中间状态。
q8, scale = quantize_int8_per_vector(kv)  # 计算并保存当前步骤的中间状态。
restored = dequantize_int8(q8, scale)  # 计算并保存当前步骤的中间状态。
mse = (kv - restored).pow(2).mean().item()  # 计算并保存当前步骤的中间状态。
main_ratio = q8.numel() * q8.element_size() / (kv.numel() * kv.element_size())  # 计算并保存当前步骤的中间状态。
actual_ratio = (q8.numel() * q8.element_size() + scale.numel() * scale.element_size()) / (kv.numel() * kv.element_size())  # 计算并保存当前步骤的中间状态。
print(f"MSE={mse:.8f}, 主数据比例={main_ratio:.3f}, 含 scale 比例={actual_ratio:.3f}")  # 计算并保存当前步骤的中间状态。
assert mse < 0.001  # 用受控断言验证关键不变量。


## 7. continuous batching 与 chunked prefill

静态 batch 等最长请求结束，短请求槽位空转。continuous batching 在 iteration 边界移出已完成请求、加入新请求，每轮每个 running 请求通常贡献一个 decode token。长 prefill 会阻塞这些延迟敏感 token，所以 chunked prefill 把 prompt 分块，用剩余 token budget 与 decode 交错。chunk 越大，prefill GEMM 越高效但 ITL 更容易抖；越小，调度和 kernel launch 开销越高。


In [ ]:
arrivals = deque([  # 计算并保存当前步骤的中间状态。
    {"id": "A", "remaining": 2},  # 执行当前语句以推进本节示例。
    {"id": "B", "remaining": 5},  # 执行当前语句以推进本节示例。
    {"id": "C", "remaining": 3},  # 执行当前语句以推进本节示例。
    {"id": "D", "remaining": 2},  # 执行当前语句以推进本节示例。
])  # 执行当前语句以推进本节示例。
running, max_batch = [], 2  # 计算并保存当前步骤的中间状态。
iteration = 0  # 计算并保存当前步骤的中间状态。
while arrivals or running:  # 在终止条件满足前持续推进状态。
    while arrivals and len(running) < max_batch:  # 在终止条件满足前持续推进状态。
        running.append(arrivals.popleft())  # 执行当前语句以推进本节示例。
    iteration += 1  # 计算并保存当前步骤的中间状态。
    print(f"iteration {iteration}: decode {[r['id'] for r in running]}")  # 执行当前语句以推进本节示例。
    for req in running:  # 遍历输入元素以累积或检查结果。
        req["remaining"] -= 1  # 计算并保存当前步骤的中间状态。
    running = [r for r in running if r["remaining"] > 0]  # 计算并保存当前步骤的中间状态。


## 8. 训练/推理、优化路线与工程坑

训练已知整段序列，可一次并行 QKV 并保留反向图；逐 token cache 会破坏并行，若 detach 又截断跨位置梯度，所以标准 teacher-forcing 训练通常禁用 cache。prefill 批量写 prompt KV、影响 TTFT；decode 读取历史并追加一个位置、常受带宽限制。

| 方法 | 主要解决 | 核心代价/风险 |
|---|---|---|
| MQA/GQA | 每 token KV 字节与带宽 | KV 表示共享、内核支持 |
| PagedAttention | 预留浪费、碎片、共享 | 间接寻址、元数据 |
| Prefix cache | 重复 prefill | 驻留容量、严格 key 与隔离 |
| KV8/KV4 | 容量和读带宽 | 量化误差、scale 与融合内核 |
| Sliding window | 单请求无限增长 | 丢失窗口外信息或需模型原生支持 |
| Offload | GPU 总容量 | PCIe/网络 stall，活跃 cache 难下沉 |
| Speculative decode | target 串行步数 | 双 cache、临时块与回滚 |

高频坑：新 position 错一位；K 重复/漏做 RoPE；seq_len 与写 slot 更新顺序错；读取预留未写槽；GQA 映射错；beam 只重排 token 不重排 table；COW 漏复制尾块；speculative 拒绝后未回滚；量化 scale 对错 block；请求释放两次或漏释放。调试应比较 full 与 cached logits，并记录 `(request, layer, logical_pos, block, offset)`。


## 9. 指标、练习与面试总结

TTFT 包含排队与 prefill；TPOT 是首 token 后平均每输出 token 时间；ITL 看相邻流式 token 间隔及尾部；吞吐要注明 input、output 还是 total tokens/s。prefix hit 主要改善 TTFT，GQA/量化常改善 TPOT 与容量，分页主要提高可接纳并发，offload 可能牺牲 ITL。

### 动手练习

1. 把教学实现的 `torch.cat` 改为预分配 buffer + 写指针，并保持 full=cached。
2. 为 `MiniPagedAllocator` 增加 release、OOM 回滚和随机状态机测试。
3. 实现 `Hq=4,Hkv=2` 的 GQA cached attention，不物理复制 KV。
4. 模拟 block size 为 4/16/64 时，真实长度分布的尾块碎片。
5. 分别量化 K、V，比较 attention 输出误差与缓存字节。
6. 给 continuous batching 加 token budget、优先级和 aging，观察吞吐—尾延迟取舍。

### 面试 60 秒主线

KV Cache 按层保存历史投影后的 K/V，新步骤只算当前 Q/K/V并让单 Q 扫历史，避免重复计算但显存和单步读取随 token 数增长。容量是 `2*L*tokens*Hkv*Dh*bytes`；GQA 降 Hkv，分页降预留与碎片，连续批处理摊销 decode 权重读取，prefix cache 复用公共块，量化降低字节。最后说明逻辑 position、block table、引用计数/COW 与 full=cached 测试是正确性核心。
